# Feature and Logit Extraction

This notebook extracts the learned representations and classification outputs from the trained ResNet-18 baseline.

For each CIFAR-10 image, we will obtain:

* **Penultimate features:** 512-dimensional representation produced by ResNet-18.
* **Logits:** 10 raw class scores produced by the final classification layer.
* **Probabilities:** normalized class probabilities obtained from the logits.

These outputs will provide the foundation for the subsequent distribution-shift, feature-distance, and calibration experiments.


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

from torchvision import datasets, transforms
from torchvision.models import resnet18
from torch.utils.data import DataLoader

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [3]:
transform = transforms.ToTensor()

test_dataset = datasets.CIFAR10(
    root="/kaggle/working/data",
    train=False,
    download=True,
    transform=transform
)

testloader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False
)

print("Test images:", len(test_dataset))

100%|██████████| 170M/170M [40:01<00:00, 71.0kB/s] 


Test images: 10000


In [5]:
import os

for root, dirs, files in os.walk("/kaggle/working"):
    for file in files:
        if file == "baseline_resnet18_cifar10.pth":
            print(os.path.join(root, file))

In [6]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith(".pth"):
            print(os.path.join(root, file))

/kaggle/input/datasets/kimi11/checkpoint/baseline_resnet18_cifar10.pth


In [7]:
import torch
import torch.nn as nn
from torchvision.models import resnet18

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Recreate the exact architecture used in Notebook 01
model = resnet18(num_classes=10)
model.conv1 = nn.Conv2d(
    3, 64, kernel_size=3, stride=1, padding=1, bias=False
)
model.maxpool = nn.Identity()

# Load the trained weights
checkpoint_path = "/kaggle/input/datasets/kimi11/checkpoint/baseline_resnet18_cifar10.pth"

state_dict = torch.load(checkpoint_path, map_location=device, weights_only=True)
model.load_state_dict(state_dict)

model = model.to(device)
model.eval()

print("Checkpoint loaded successfully.")
print("Device:", device)

Checkpoint loaded successfully.
Device: cuda


In [8]:
feature_extractor = nn.Sequential(
    *list(model.children())[:-1]
).to(device)

feature_extractor.eval()

print("Feature extractor ready.")

Feature extractor ready.


In [9]:
import numpy as np
import torch

all_features = []
all_logits = []
all_labels = []

model.eval()
feature_extractor.eval()

with torch.no_grad():
    for images, labels in testloader:
        images = images.to(device)

        # Extract 512-dimensional features
        features = feature_extractor(images)
        features = torch.flatten(features, start_dim=1)

        # Extract 10 raw class scores (logits)
        logits = model(images)

        all_features.append(features.cpu())
        all_logits.append(logits.cpu())
        all_labels.append(labels)

# Combine every batch into complete datasets
all_features = torch.cat(all_features, dim=0).numpy()
all_logits = torch.cat(all_logits, dim=0).numpy()
all_labels = torch.cat(all_labels, dim=0).numpy()

print("Features shape:", all_features.shape)
print("Logits shape:", all_logits.shape)
print("Labels shape:", all_labels.shape)

Features shape: (10000, 512)
Logits shape: (10000, 10)
Labels shape: (10000,)


features are internal representations; logits are the model's raw class scores. We will use both in the subsequent distribution-shift and calibration experiments.

In [10]:
import torch
import numpy as np

# Convert logits into probabilities using Softmax
all_probabilities = torch.softmax(
    torch.tensor(all_logits), dim=1
).numpy()

print("Probabilities shape:", all_probabilities.shape)
print("First image probabilities:", all_probabilities[0])
print("Sum of probabilities:", all_probabilities[0].sum())

Probabilities shape: (10000, 10)
First image probabilities: [1.6140564e-03 1.6003149e-03 5.5552919e-05 9.7393596e-01 5.2094601e-05
 1.9321641e-02 2.2652701e-03 3.1333566e-05 1.0375193e-03 8.6261825e-05]
Sum of probabilities: 1.0


In [11]:
import os
import numpy as np

save_dir = "/kaggle/working/fdcc/results"
os.makedirs(save_dir, exist_ok=True)

np.savez_compressed(
    os.path.join(save_dir, "cifar10_test_features_logits.npz"),
    features=all_features,
    logits=all_logits,
    probabilities=all_probabilities,
    labels=all_labels
)

print("Results saved successfully.")
print("File:", os.path.join(save_dir, "cifar10_test_features_logits.npz"))
print(f"Features: {all_features.shape}")
print(f"Logits: {all_logits.shape}")
print(f"Probabilities: {all_probabilities.shape}")
print(f"Labels: {all_labels.shape}")

Results saved successfully.
File: /kaggle/working/fdcc/results/cifar10_test_features_logits.npz
Features: (10000, 512)
Logits: (10000, 10)
Probabilities: (10000, 10)
Labels: (10000,)


Yes. The `cifar10_test_features_logits.npz` file contains the extracted results for all 10,000 CIFAR-10 test images combined.

It contains four arrays:

| Array           | Shape          | What it stores                   |
| --------------- | -------------- | -------------------------------- |
| `features`      | `(10000, 512)` | 512 features per image           |
| `logits`        | `(10000, 10)`  | 10 raw class scores per image    |
| `probabilities` | `(10000, 10)`  | 10 class probabilities per image |
| `labels`        | `(10000,)`     | One correct label per image      |

Each row corresponds to the same image across all four arrays. For example, `features[0]`, `logits[0]`, `probabilities[0]`, and `labels[0]` all refer to the first test image.


In [12]:
import numpy as np
import os

results_path = "/kaggle/working/fdcc/results/cifar10_test_features_logits.npz"

data = np.load(results_path)

print("File exists:", os.path.exists(results_path))
print("Arrays saved:", data.files)
print("Features:", data["features"].shape)
print("Logits:", data["logits"].shape)
print("Probabilities:", data["probabilities"].shape)
print("Labels:", data["labels"].shape)
print("Probabilities sum to 1:", np.allclose(data["probabilities"].sum(axis=1), 1.0))

File exists: True
Arrays saved: ['features', 'logits', 'probabilities', 'labels']
Features: (10000, 512)
Logits: (10000, 10)
Probabilities: (10000, 10)
Labels: (10000,)
Probabilities sum to 1: True


## Experiment 2: Feature and Logit Extraction — Conclusion

In this experiment, we loaded the trained ResNet-18 baseline model using its saved checkpoint and used it to process all 10,000 CIFAR-10 test images.

For each image, we extracted its **512-dimensional penultimate feature representation**, **10 raw classification logits**, **10 softmax probabilities**, and corresponding ground-truth label. The extracted results were combined into a single compressed `.npz` file, `cifar10_test_features_logits.npz`, for reuse in subsequent experiments.

This experiment establishes the data foundation for the next stages of FDCC. The feature representations will be used to investigate feature-space displacement under distribution shift, while logits and probabilities will support the analysis of predictive confidence and calibration.

**Outcome:** Feature and logit extraction was completed for the entire test set, providing the representations and predictions required for the subsequent distribution-shift and calibration experiments.
